# Comparative Evaluation of GA and PSO Feature Selection for Heart Disease Prediction — Workflow V2

This notebook implements the verified repeated nested-validation design and writes only to a new versioned output directory. The previous run `final_3a9840d321c5832c` is a locked empirical record and is never modified. Checksums establish file identity, not data correctness, representativeness, clinical validity or external validity. Integrity PASS rows are recorded workflow assertions rather than an independent code audit. The analysis is an internal methodological benchmark; it does not establish causality, clinical readiness or diagnostic replacement.


## 1. Installation and imports

Run the installation cell once in a fresh Jupyter environment. Versions are bounded for compatibility; the exact versions used in a run are captured later. Known numerical warnings are handled locally rather than globally hidden.

In [ ]:
#%pip install -q "numpy>=2.0,<3" "pandas>=2.2,<3" "scipy>=1.13,<2" "scikit-learn>=1.6,<2" "xgboost>=2.0,<4" "pygad>=3.3,<4" "pyswarms>=1.3,<2" "matplotlib>=3.8,<4" "seaborn>=0.13,<1" "joblib>=1.4,<2"


In [ ]:
from __future__ import annotations

import hashlib
import importlib.metadata as metadata
import json
import math
import os
import platform
import random
import time
import urllib.request
import warnings
from contextlib import contextmanager
import zipfile
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from itertools import combinations
from pathlib import Path
from tempfile import TemporaryDirectory

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pygad
import pyswarms as ps
import seaborn as sns
import sklearn
import xgboost
from scipy.special import expit, logit
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, brier_score_loss,
    confusion_matrix, f1_score, matthews_corrcoef,
    precision_recall_curve, roc_auc_score, roc_curve,
)
from sklearn.model_selection import (
    GridSearchCV, RepeatedStratifiedKFold, StratifiedKFold,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)

## 2. Configuration

`FAST_MODE=True` executes one complete outer split across all 15 conditions with small optimiser and bootstrap budgets. It validates the workflow but must not be labelled as the final dissertation analysis. Set `FAST_MODE=False` and `ALLOW_FINAL_RUN=True` only after the smoke tests pass and the final computational budget has been frozen without consulting outer-test results.

In [ ]:
@dataclass(frozen=True)
class Config:
    workflow_version: str = "2.0.0"
    fast_mode: bool = True
    allow_final_run: bool = False
    random_state: int = 20260817
    outer_folds: int = 5
    outer_repeats: int = 2
    inner_folds: int = 3
    optimiser_seeds: tuple = (101,) if True else (101, 202, 303)
    fitness_eval_budget: int = 12 if True else 160
    bootstrap_replicates: int = 100 if True else 2000
    decision_threshold: float = 0.50
    n_jobs_outer: int = 1
    output_root: str = "workflow_outputs"
    checkpoint_dir: str = "workflow_outputs/checkpoints"
    source_url: str = "https://archive.ics.uci.edu/static/public/45/heart+disease.zip"
    source_member: str = "processed.cleveland.data"
    local_data_file: str = "data/processed.cleveland.data"
    expected_sha256: str = "a74b7efa387bc9d108d7d0115d831fe9b414b29ae7124f331b622b4efa0427c8"
    ga_population: int = 6 if True else 16
    pso_particles: int = 6 if True else 16
    fitness_tolerance: float = 1e-6


# Change only these two values for the frozen final analysis.
#FAST_MODE = True
#ALLOW_FINAL_RUN = False

FAST_MODE = False
ALLOW_FINAL_RUN = True

CONFIG = Config(
    fast_mode=FAST_MODE,
    allow_final_run=ALLOW_FINAL_RUN,
    optimiser_seeds=(101,) if FAST_MODE else (101, 202, 303),
    fitness_eval_budget=12 if FAST_MODE else 160,
    bootstrap_replicates=100 if FAST_MODE else 2000,
    ga_population=6 if FAST_MODE else 16,
    pso_particles=6 if FAST_MODE else 16,
)

if CONFIG.fast_mode and CONFIG.allow_final_run:
    raise RuntimeError("Final analysis cannot run while FAST_MODE=True.")
if not CONFIG.fast_mode and not CONFIG.allow_final_run:
    raise RuntimeError("Set ALLOW_FINAL_RUN=True explicitly for the final analysis.")
if CONFIG.fitness_eval_budget % CONFIG.pso_particles != 0:
    raise ValueError("FITNESS_EVAL_BUDGET must be divisible by PSO particle count.")

CLASSIFIERS = ["LR", "DT", "RF", "SVM", "XGB"]
CONDITIONS = ["ALL", "GA", "PSO"]
SELECTORS = ["GA", "PSO"]
FEATURES = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
    "thalach", "exang", "oldpeak", "slope", "ca", "thal",
]
TARGET = "num"
NUMERIC_FEATURES = ["age", "trestbps", "chol", "thalach", "oldpeak"]
CATEGORICAL_FEATURES = [
    "sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal",
]
ALL_COLUMNS = FEATURES + [TARGET]

outer_splits_to_run = 1 if CONFIG.fast_mode else CONFIG.outer_folds * CONFIG.outer_repeats
units = outer_splits_to_run * len(CLASSIFIERS)
upper_fitness_calls = units * len(SELECTORS) * len(CONFIG.optimiser_seeds) * CONFIG.fitness_eval_budget
print(f"Mode: {'FAST smoke test' if CONFIG.fast_mode else 'FINAL'}")
print(f"Outer classifier units: {units}")
print(f"Upper bound on optimiser mask requests: {upper_fitness_calls:,}")

## 3. Reproducibility and file utilities

Configuration fingerprints prevent incompatible checkpoints from being combined. JSON conversion retains full machine-readable precision.

In [ ]:
def set_global_seed(seed: int) -> None:
    """Seed Python and NumPy for repeatable stochastic operations."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)


def json_ready(value):
    """Convert common scientific-Python objects to JSON-safe values."""
    if isinstance(value, dict):
        return {str(k): json_ready(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_ready(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    return value


def save_json(data, path: Path) -> None:
    """Save JSON atomically enough for a single-process notebook run."""
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(json_ready(data), indent=2, sort_keys=True), encoding="utf-8")
    temp.replace(path)


def sha256_file(path: Path) -> str:
    """Return a file's SHA-256 digest."""
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def make_config_fingerprint(data) -> str:
    """Create a stable short fingerprint from JSON-serialisable settings."""
    payload = json.dumps(json_ready(data), sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:16]


def package_versions() -> dict:
    """Capture Python, platform and analytical package versions."""
    packages = [
        "numpy", "pandas", "scipy", "scikit-learn", "xgboost", "pygad",
        "pyswarms", "matplotlib", "seaborn", "joblib",
    ]
    versions = {name: metadata.version(name) for name in packages}
    versions.update({"python": platform.python_version(), "platform": platform.platform()})
    return versions


set_global_seed(CONFIG.random_state)
CONFIG_FINGERPRINT = make_config_fingerprint(asdict(CONFIG))
RUN_NAME = f"{'fast_v2' if CONFIG.fast_mode else 'final_v2'}_{CONFIG_FINGERPRINT}"
OUTPUT_DIR = Path(CONFIG.output_root) / RUN_NAME
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"
CHECKPOINT_DIR = Path(CONFIG.checkpoint_dir) / RUN_NAME
for folder in [OUTPUT_DIR, TABLE_DIR, FIGURE_DIR, CHECKPOINT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

environment = package_versions()
save_json(environment, OUTPUT_DIR / "software_versions.json")
save_json(asdict(CONFIG), OUTPUT_DIR / "configuration.json")
print(json.dumps(environment, indent=2))
print("Configuration fingerprint:", CONFIG_FINGERPRINT)

CAPTURED_WARNINGS = []

@contextmanager
def capture_warnings(section: str, operation: str, severity: str = "analytical"):
    """Capture warnings with provenance; never hide them globally."""
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        yield
    for item in caught:
        CAPTURED_WARNINGS.append({
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
            "category": item.category.__name__,
            "message": str(item.message),
            "section": section,
            "operation": operation,
            "severity": severity,
            "disposition": "recorded; execution continued",
        })


def artifact_metadata(path: Path) -> dict:
    """Describe one canonical or alias artifact for the final manifest."""
    rel = str(path.relative_to(OUTPUT_DIR))
    suffix = path.suffix.lower()
    artifact_type = {".csv": "table", ".json": "machine-readable record", ".png": "figure"}.get(suffix, "artifact")
    producer = "figures" if "figures" in path.parts else "tables" if "tables" in path.parts else "workflow"
    alias_of = "tuning_results.json" if rel.replace("\\", "/") == "tables/best_hyperparameters_by_outer_split.json" else None
    grain_map = {
        "all_outer_predictions.csv": "observation × outer split × classifier × condition",
        "aggregated_oof_predictions.csv": "observation × classifier × condition",
        "fold_metrics.csv": "outer split × classifier × condition",
        "all_optimiser_runs.json": "outer split × classifier × selector × seed",
        "selected_masks.json": "outer split × classifier × condition",
        "tuning_results.json": "outer split × classifier × condition",
        "paired_bootstrap_differences.csv": "classifier × comparison × metric",
    }
    return {
        "relative_path": rel,
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
        "artifact_type": artifact_type,
        "producing_section": producer,
        "row_or_key_grain": grain_map.get(path.name, "documented by producing table or record"),
        "precision_policy": "display-rounded" if "display_" in path.name else "full precision or lossless",
        "validation_rule": "size and SHA-256 verified after creation",
        "canonical": alias_of is None,
        "alias_of": alias_of,
    }


## 4. Data acquisition, provenance and verification

Only UCI's processed Cleveland member is accepted. The loader records the retrieval time and checksum, treats `?` and blank strings as missing, verifies 303 rows × 14 columns and recodes `num` as `0` versus `1–4`. A changed checksum or incompatible schema stops execution.

Dataset: UCI Machine Learning Repository, *Heart Disease* (1988), https://doi.org/10.24432/C52P4X.

In [ ]:
def acquire_cleveland(config: Config) -> tuple[Path, str]:
    """Use the local verified file or download only UCI's Cleveland member."""
    path = Path(config.local_data_file)
    if path.exists():
        origin = "verified local copy supplied with notebook"
    else:
        path.parent.mkdir(parents=True, exist_ok=True)
        with TemporaryDirectory() as temp_dir:
            archive = Path(temp_dir) / "heart_disease.zip"
            urllib.request.urlretrieve(config.source_url, archive)
            with zipfile.ZipFile(archive) as zipped:
                if config.source_member not in zipped.namelist():
                    raise RuntimeError("UCI archive does not contain processed.cleveland.data.")
                path.write_bytes(zipped.read(config.source_member))
        origin = config.source_url
    observed_hash = sha256_file(path)
    if observed_hash != config.expected_sha256:
        raise RuntimeError(
            "Cleveland file checksum differs from the frozen analysis version. "
            f"Observed {observed_hash}; expected {config.expected_sha256}."
        )
    return path, origin


def validate_dataset_schema(frame: pd.DataFrame) -> None:
    """Fail early if the frozen Cleveland schema or target coding is incompatible."""
    if frame.shape != (303, 14):
        raise ValueError(f"Expected Cleveland shape (303, 14), observed {frame.shape}.")
    if list(frame.columns) != ALL_COLUMNS:
        raise ValueError("Unexpected column names or order.")
    if frame[TARGET].isna().any():
        raise ValueError("Target contains missing values.")
    target_values = set(frame[TARGET].astype(int).unique())
    if not target_values.issubset({0, 1, 2, 3, 4}):
        raise ValueError(f"Unexpected original target values: {sorted(target_values)}")
    if len(FEATURES) != 13 or len(set(FEATURES)) != 13:
        raise ValueError("The analysis must contain 13 unique original predictors.")


data_path, data_origin = acquire_cleveland(CONFIG)
raw = pd.read_csv(
    data_path, header=None, names=ALL_COLUMNS,
    na_values=["?", "", "NA", "N/A"], skipinitialspace=True,
)
raw = raw.apply(pd.to_numeric, errors="coerce")
validate_dataset_schema(raw)

X = raw[FEATURES].copy()
y_original = raw[TARGET].astype(int)
y = (y_original > 0).astype(int).rename("heart_disease")
X.index.name = "row_id"
y.index = X.index

provenance = {
    "source_url": CONFIG.source_url,
    "origin_used": data_origin,
    "retrieved_or_verified_utc": datetime.now(timezone.utc).isoformat(),
    "local_filename": str(data_path),
    "sha256": sha256_file(data_path),
    "raw_shape": list(raw.shape),
    "analytical_shape": [len(X), len(FEATURES)],
    "target_rule": "0=no heart disease; original values 1-4=heart disease",
    "original_target_counts": y_original.value_counts().sort_index().to_dict(),
    "binary_target_counts": y.value_counts().sort_index().to_dict(),
    "predictors": FEATURES,
    "numeric_predictors": NUMERIC_FEATURES,
    "categorical_predictors": CATEGORICAL_FEATURES,
    "duplicate_rows_retained": int(raw.duplicated().sum()),
}
save_json(provenance, OUTPUT_DIR / "data_provenance.json")
display(pd.Series(provenance, name="value").to_frame())

## 5. Focused exploratory analysis

EDA is descriptive only and does not determine feature selection, tuning or threshold choices. Duplicate rows are counted but not silently removed.

In [ ]:
missingness = pd.DataFrame({
    "missing_n": X.isna().sum(),
    "missing_percent": X.isna().mean().mul(100),
}).rename_axis("feature").reset_index()
class_table = y.value_counts().sort_index().rename_axis("class").reset_index(name="n")
class_table["label"] = class_table["class"].map({0: "No disease", 1: "Disease"})
numeric_summary = X[NUMERIC_FEATURES].describe().T.reset_index(names="feature")
categorical_frequencies = pd.concat(
    [
        X[name].value_counts(dropna=False).rename_axis("value").reset_index(name="n")
        .assign(feature=name)
        for name in CATEGORICAL_FEATURES
    ],
    ignore_index=True,
)[["feature", "value", "n"]]

class_table.to_csv(TABLE_DIR / "dataset_class_distribution.csv", index=False)
missingness.to_csv(TABLE_DIR / "dataset_missingness.csv", index=False)
numeric_summary.to_csv(TABLE_DIR / "dataset_numeric_summary.csv", index=False)
categorical_frequencies.to_csv(TABLE_DIR / "dataset_categorical_frequencies.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.barplot(data=class_table, x="label", y="n", hue="label", legend=False, ax=axes[0])
axes[0].set(title="Binary outcome distribution", xlabel="", ylabel="Observations")
sns.barplot(data=missingness, x="missing_percent", y="feature", color="#4C78A8", ax=axes[1])
axes[1].set(title="Missingness by predictor", xlabel="Missing (%)", ylabel="")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "01_class_and_missingness.png", dpi=300, bbox_inches="tight")
plt.show()
display(missingness, numeric_summary)

## 6. Original-feature masks and fold-local pipelines

Every GA chromosome and PSO particle has exactly 13 bits. One-hot encoding occurs only after the mask has selected original variables. Dense encoded matrices are used because the dataset is small and this is compatible with every approved estimator.

In [ ]:
def validate_mask(mask, allow_empty: bool = False) -> tuple[int, ...]:
    """Validate and return a 13-bit original-feature mask."""
    cleaned = tuple(int(v) for v in np.asarray(mask).round().astype(int).tolist())
    if len(cleaned) != len(FEATURES):
        raise ValueError(f"Mask must have {len(FEATURES)} bits, not {len(cleaned)}.")
    if not set(cleaned).issubset({0, 1}):
        raise ValueError("Mask values must be binary.")
    if not allow_empty and sum(cleaned) == 0:
        raise ValueError("Empty feature masks cannot be fitted.")
    return cleaned


def mask_to_features(mask) -> list[str]:
    """Translate a valid mask to original predictor names."""
    cleaned = validate_mask(mask)
    return [name for name, keep in zip(FEATURES, cleaned) if keep]


def get_base_estimator(classifier_name: str, seed: int):
    """Return the fixed classifier used only inside wrapper fitness."""
    estimators = {
        "LR": LogisticRegression(C=1.0, solver="liblinear", max_iter=2000, random_state=seed),
        "DT": DecisionTreeClassifier(max_depth=4, min_samples_leaf=5, random_state=seed),
        "RF": RandomForestClassifier(
            n_estimators=100, max_depth=5, min_samples_leaf=2,
            random_state=seed, n_jobs=1,
        ),
        "SVM": SVC(C=1.0, gamma="scale", kernel="rbf", probability=False, random_state=seed),
        "XGB": XGBClassifier(
            n_estimators=100, max_depth=3, learning_rate=0.05,
            objective="binary:logistic", eval_metric="logloss",
            random_state=seed, n_jobs=1, verbosity=0,
        ),
    }
    return estimators[classifier_name]


def build_pipeline(classifier_name: str, selected_features: list[str], estimator) -> Pipeline:
    """Build imputation/encoding and classifier steps for selected original predictors."""
    selected_numeric = [f for f in NUMERIC_FEATURES if f in selected_features]
    selected_categorical = [f for f in CATEGORICAL_FEATURES if f in selected_features]
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if classifier_name in {"LR", "SVM"}:
        numeric_steps.append(("scaler", StandardScaler()))
    transformers = []
    if selected_numeric:
        transformers.append(("numeric", Pipeline(numeric_steps), selected_numeric))
    if selected_categorical:
        categorical_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ])
        transformers.append(("categorical", categorical_pipe, selected_categorical))
    preprocessor = ColumnTransformer(transformers, remainder="drop", sparse_threshold=0)
    return Pipeline([("preprocess", preprocessor), ("model", estimator)])


assert len(validate_mask(np.ones(13))) == 13
assert mask_to_features(np.ones(13)) == FEATURES
try:
    validate_mask(np.zeros(13))
    raise AssertionError("Empty mask test failed.")
except ValueError:
    pass

test_pipe = build_pipeline("LR", ["age", "cp"], get_base_estimator("LR", 1))
test_pipe.fit(X[["age", "cp"]].iloc[:100], y.iloc[:100])
encoded_width = test_pipe.named_steps["preprocess"].transform(X[["age", "cp"]].iloc[:5]).shape[1]
assert encoded_width >= 2 and len(FEATURES) == 13
print("Mask and pipeline tests passed; encoded width is independent of mask length.")

## 7. Frozen outer and inner splits

The 10 outer splits are generated once and shared by all 15 conditions. Inner index arrays are created once per outer-training set and reused for baseline tuning, GA/PSO fitness and subset-specific tuning.

In [ ]:
def index_hash(indices) -> str:
    """Hash an integer index array for split auditing."""
    return hashlib.sha256(np.asarray(indices, dtype=np.int64).tobytes()).hexdigest()[:16]


def make_outer_splits(X_data, y_data, config: Config):
    """Generate the fixed repeated stratified outer splits and audit metadata."""
    splitter = RepeatedStratifiedKFold(
        n_splits=config.outer_folds,
        n_repeats=config.outer_repeats,
        random_state=config.random_state,
    )
    output = []
    for split_id, (train_idx, test_idx) in enumerate(splitter.split(X_data, y_data)):
        assert split_id // config.outer_folds < config.outer_repeats
        assert split_id % config.outer_folds in range(config.outer_folds)
        assert not set(train_idx).intersection(test_idx)
        output.append({
            "split_id": split_id,
            "repeat_id": split_id // config.outer_folds,
            "fold_id": split_id % config.outer_folds,
            "train_idx": train_idx,
            "test_idx": test_idx,
            "train_hash": index_hash(train_idx),
            "test_hash": index_hash(test_idx),
        })
    return output


def make_inner_splits(y_outer_train: pd.Series, config: Config):
    """Create explicit frozen inner indices relative to the outer-training set."""
    splitter = StratifiedKFold(
        n_splits=config.inner_folds, shuffle=True, random_state=config.random_state,
    )
    dummy = np.zeros(len(y_outer_train))
    splits = [(tr.copy(), va.copy()) for tr, va in splitter.split(dummy, y_outer_train)]
    for train_idx, valid_idx in splits:
        assert not set(train_idx).intersection(valid_idx)
        assert y_outer_train.iloc[train_idx].nunique() == 2
        assert y_outer_train.iloc[valid_idx].nunique() == 2
    return splits


outer_splits = make_outer_splits(X, y, CONFIG)
split_metadata = pd.DataFrame([
    {k: v for k, v in split.items() if not k.endswith("_idx")}
    for split in outer_splits
])
split_metadata.to_csv(OUTPUT_DIR / "outer_split_metadata.csv", index=False)
assert len(outer_splits) == CONFIG.outer_folds * CONFIG.outer_repeats
display(split_metadata)

## 8. Restrained classifier tuning grids

The same classifier-specific grid is used for ALL, GA and PSO. FAST mode uses one setting to test execution; final mode uses the frozen restrained grids below. XGBoost and all inner operations use one job to avoid nested oversubscription.

In [ ]:
def get_param_grid(classifier_name: str, fast_mode: bool) -> dict:
    """Return the pre-specified model grid with pipeline prefixes."""
    full = {
        "LR": {"model__C": [0.1, 1.0, 10.0]},
        "DT": {"model__max_depth": [None, 3, 5], "model__min_samples_leaf": [1, 5, 10]},
        "RF": {
            "model__n_estimators": [150, 300], "model__max_depth": [None, 5],
            "model__min_samples_leaf": [1, 5],
        },
        "SVM": {"model__C": [0.1, 1.0, 10.0], "model__gamma": ["scale", 0.01, 0.1]},
        "XGB": {
            "model__n_estimators": [100, 200], "model__max_depth": [2, 3],
            "model__learning_rate": [0.03, 0.1],
        },
    }
    if not fast_mode:
        return full[classifier_name]
    base = get_base_estimator(classifier_name, CONFIG.random_state).get_params()
    keys = full[classifier_name].keys()
    return {key: [base[key.replace("model__", "")]] for key in keys}


grid_inventory = []
for classifier in CLASSIFIERS:
    for parameter, values in get_param_grid(classifier, CONFIG.fast_mode).items():
        grid_inventory.append({"classifier": classifier, "parameter": parameter, "values": values})
grid_table = pd.DataFrame(grid_inventory)
grid_table.to_csv(TABLE_DIR / "model_tuning_grids.csv", index=False)
display(grid_table)

## 9. Classifier-specific fitness evaluator and deterministic cache

The cache key contains outer split, classifier, mask, base-estimator settings, preprocessing metadata and frozen inner splits. It deliberately excludes optimiser and seed. A run-level `seen` set makes the common evaluation cap apply to unique requested masks even when a global cache hit avoids repeated fitting.

In [ ]:
FITNESS_ERRORS = []


def estimator_fingerprint(estimator) -> str:
    """Fingerprint an estimator's complete parameter dictionary."""
    return make_config_fingerprint(estimator.get_params(deep=True))


def inner_split_fingerprint(inner_splits) -> str:
    """Fingerprint explicit frozen inner indices."""
    representation = [
        {"train": index_hash(train), "valid": index_hash(valid)}
        for train, valid in inner_splits
    ]
    return make_config_fingerprint(representation)


def evaluate_mask(
    mask, X_outer_train, y_outer_train, inner_splits, classifier_name,
    base_estimator, cache, context, run_state,
) -> float:
    """Return mean inner ROC-AUC for one original-feature mask."""
    run_state["attempted"] += 1
    cleaned = validate_mask(mask, allow_empty=True)
    if sum(cleaned) == 0:
        run_state["empty_masks"] += 1
        return 0.0
    if cleaned not in run_state["seen"]:
        if len(run_state["seen"]) >= run_state["budget"]:
            run_state["budget_rejections"] += 1
            return 0.0
        run_state["seen"].add(cleaned)

    cache_key = (
        context["outer_split"], classifier_name, cleaned,
        context["estimator_fingerprint"], context["preprocessing_fingerprint"],
        context["inner_split_fingerprint"],
    )
    if cache_key in cache:
        run_state["cache_hits"] += 1
        return cache[cache_key]

    selected = mask_to_features(cleaned)
    scores = []
    try:
        for train_idx, valid_idx in inner_splits:
            pipeline = build_pipeline(classifier_name, selected, clone(base_estimator))
            pipeline.fit(X_outer_train.iloc[train_idx][selected], y_outer_train.iloc[train_idx])
            if classifier_name == "SVM":
                prediction = pipeline.decision_function(X_outer_train.iloc[valid_idx][selected])
            else:
                prediction = pipeline.predict_proba(X_outer_train.iloc[valid_idx][selected])[:, 1]
            scores.append(roc_auc_score(y_outer_train.iloc[valid_idx], prediction))
        fitness = float(np.mean(scores))
    except Exception as exc:
        fitness = 0.0
        FITNESS_ERRORS.append({
            "outer_split": context["outer_split"], "classifier": classifier_name,
            "mask": cleaned, "error_type": type(exc).__name__, "message": str(exc),
        })
    cache[cache_key] = fitness
    run_state["cache_misses"] += 1
    return fitness


def new_run_state(budget: int) -> dict:
    """Return transparent counters for one optimiser seed run."""
    return {
        "budget": budget, "seen": set(), "attempted": 0, "cache_hits": 0,
        "cache_misses": 0, "empty_masks": 0, "budget_rejections": 0,
    }

## 10. Genetic Algorithm wrapper

PyGAD uses a 13-bit chromosome. All seed runs are retained; the one passed to tuning is later chosen only by inner fitness, then feature count within `1e-6`, then seed.

In [ ]:
def run_ga(
    X_train, y_train, inner_splits, classifier_name, split_id,
    seed, cache, config: Config,
) -> dict:
    """Run one budgeted classifier-specific binary GA seed."""
    set_global_seed(seed)
    estimator = get_base_estimator(classifier_name, config.random_state)
    context = {
        "outer_split": split_id,
        "estimator_fingerprint": estimator_fingerprint(estimator),
        "preprocessing_fingerprint": make_config_fingerprint({
            "numeric": NUMERIC_FEATURES, "categorical": CATEGORICAL_FEATURES,
            "scaling": classifier_name in {"LR", "SVM"}, "dense": True,
        }),
        "inner_split_fingerprint": inner_split_fingerprint(inner_splits),
    }
    state = new_run_state(config.fitness_eval_budget)
    rng = np.random.default_rng(seed)
    initial = rng.integers(0, 2, size=(config.ga_population, len(FEATURES)))
    for row in initial:
        if row.sum() == 0:
            row[rng.integers(0, len(FEATURES))] = 1

    def fitness_function(ga_instance, solution, solution_index):
        return evaluate_mask(
            solution, X_train, y_train, inner_splits, classifier_name,
            estimator, cache, context, state,
        )

    def stop_at_budget(ga_instance):
        return "stop" if len(state["seen"]) >= state["budget"] else None

    generations = max(1, math.ceil(config.fitness_eval_budget / config.ga_population))
    start = time.perf_counter()
    ga = pygad.GA(
        num_generations=generations,
        num_parents_mating=max(2, config.ga_population // 2),
        initial_population=initial,
        fitness_func=fitness_function,
        gene_type=int,
        gene_space=[0, 1],
        parent_selection_type="tournament",
        K_tournament=3,
        crossover_type="single_point",
        crossover_probability=0.8,
        mutation_type="random",
        mutation_probability=0.1,
        mutation_by_replacement=True,
        random_mutation_min_val=0,
        random_mutation_max_val=2,
        keep_elitism=1,
        on_generation=stop_at_budget,
        random_seed=seed,
        save_best_solutions=True,
        suppress_warnings=True,
    )
    ga.run()
    solution, fitness, _ = ga.best_solution()
    mask = validate_mask(solution)
    elapsed = time.perf_counter() - start
    return {
        "outer_split": split_id, "classifier": classifier_name, "selector": "GA",
        "seed": seed, "mask": mask, "features": mask_to_features(mask),
        "fitness": float(fitness), "feature_count": int(sum(mask)),
        "population_size": config.ga_population, "max_generations": generations,
        "parent_selection": "tournament", "crossover": "single_point:0.8",
        "mutation": "random:0.1", "runtime_seconds": elapsed,
        "attempted_evaluations": state["attempted"],
        "unique_requested": len(state["seen"]), "cache_hits": state["cache_hits"],
        "cache_misses": state["cache_misses"],
        "termination": "fitness budget reached" if len(state["seen"]) >= state["budget"] else "generation limit",
        "convergence": [float(v) for v in ga.best_solutions_fitness],
    }

## 11. Binary Particle Swarm Optimisation wrapper

PySwarms minimises cost, so the objective is `-mean inner ROC-AUC`. PSO receives the same seed count and maximum unique-mask request budget as GA. Generations and iterations are not treated as equivalent.

In [ ]:
def run_binary_pso(
    X_train, y_train, inner_splits, classifier_name, split_id,
    seed, cache, config: Config,
) -> dict:
    """Run one budgeted classifier-specific BinaryPSO seed."""
    set_global_seed(seed)
    estimator = get_base_estimator(classifier_name, config.random_state)
    context = {
        "outer_split": split_id,
        "estimator_fingerprint": estimator_fingerprint(estimator),
        "preprocessing_fingerprint": make_config_fingerprint({
            "numeric": NUMERIC_FEATURES, "categorical": CATEGORICAL_FEATURES,
            "scaling": classifier_name in {"LR", "SVM"}, "dense": True,
        }),
        "inner_split_fingerprint": inner_split_fingerprint(inner_splits),
    }
    state = new_run_state(config.fitness_eval_budget)

    def objective(particles):
        fitness = [
            evaluate_mask(
                particle, X_train, y_train, inner_splits, classifier_name,
                estimator, cache, context, state,
            )
            for particle in particles
        ]
        return -np.asarray(fitness, dtype=float)

    rng = np.random.default_rng(seed)
    initial = rng.integers(0, 2, size=(config.pso_particles, len(FEATURES)))
    for row in initial:
        if row.sum() == 0:
            row[rng.integers(0, len(FEATURES))] = 1
    options = {"c1": 1.5, "c2": 1.5, "w": 0.7, "k": min(3, config.pso_particles), "p": 2}
    iterations = config.fitness_eval_budget // config.pso_particles
    start = time.perf_counter()
    optimiser = ps.discrete.BinaryPSO(
        n_particles=config.pso_particles,
        dimensions=len(FEATURES), options=options, init_pos=initial,
    )
    best_cost, best_position = optimiser.optimize(objective, iters=iterations, verbose=False)
    mask = validate_mask(best_position)
    elapsed = time.perf_counter() - start
    return {
        "outer_split": split_id, "classifier": classifier_name, "selector": "PSO",
        "seed": seed, "mask": mask, "features": mask_to_features(mask),
        "fitness": float(-best_cost), "feature_count": int(sum(mask)),
        "swarm_size": config.pso_particles, "max_iterations": iterations,
        "c1": options["c1"], "c2": options["c2"], "w": options["w"],
        "neighbourhood_k": options["k"], "distance_p": options["p"],
        "runtime_seconds": elapsed, "attempted_evaluations": state["attempted"],
        "unique_requested": len(state["seen"]), "cache_hits": state["cache_hits"],
        "cache_misses": state["cache_misses"],
        "termination": "fixed matched evaluation batches",
        "convergence": [float(-v) for v in optimiser.cost_history],
    }


def choose_best_run(runs: list[dict], tolerance: float) -> dict:
    """Choose by inner fitness, parsimony within tolerance, then lower seed."""
    best_fitness = max(run["fitness"] for run in runs)
    tied = [run for run in runs if best_fitness - run["fitness"] <= tolerance]
    return sorted(tied, key=lambda run: (run["feature_count"], run["seed"]))[0]

## 12. Optimiser and cache smoke tests

These lightweight checks verify the budget, cache context and non-empty masks before the main loop. The complete FAST-mode outer unit is executed in Section 14.

In [ ]:
smoke_split = outer_splits[0]
smoke_train = smoke_split["train_idx"]
X_smoke, y_smoke = X.iloc[smoke_train], y.iloc[smoke_train]
inner_smoke = make_inner_splits(y_smoke, CONFIG)
smoke_cache = {}

# Independent synthetic-data smoke test with the same 13-feature schema.
synthetic_values, synthetic_target = make_classification(
    n_samples=120, n_features=13, n_informative=6, n_redundant=2,
    random_state=CONFIG.random_state,
)
X_synthetic = pd.DataFrame(synthetic_values, columns=FEATURES)
for feature in CATEGORICAL_FEATURES:
    X_synthetic[feature] = pd.qcut(
        X_synthetic[feature], q=3, labels=False, duplicates="drop"
    ).astype(float)
y_synthetic = pd.Series(synthetic_target, name="heart_disease")
synthetic_inner = make_inner_splits(y_synthetic, CONFIG)
synthetic_cache = {}
synthetic_ga = run_ga(
    X_synthetic, y_synthetic, synthetic_inner, "LR", "synthetic", 101,
    synthetic_cache, CONFIG,
)
synthetic_pso = run_binary_pso(
    X_synthetic, y_synthetic, synthetic_inner, "LR", "synthetic", 101,
    synthetic_cache, CONFIG,
)
assert sum(synthetic_ga["mask"]) > 0 and sum(synthetic_pso["mask"]) > 0
assert synthetic_ga["unique_requested"] <= CONFIG.fitness_eval_budget
assert synthetic_pso["unique_requested"] <= CONFIG.fitness_eval_budget

ga_smoke = run_ga(
    X_smoke, y_smoke, inner_smoke, "LR", 0, 101, smoke_cache, CONFIG,
)
pso_smoke = run_binary_pso(
    X_smoke, y_smoke, inner_smoke, "LR", 0, 101, smoke_cache, CONFIG,
)
assert len(ga_smoke["mask"]) == len(FEATURES) and sum(ga_smoke["mask"]) > 0
assert len(pso_smoke["mask"]) == len(FEATURES) and sum(pso_smoke["mask"]) > 0
assert ga_smoke["unique_requested"] <= CONFIG.fitness_eval_budget
assert pso_smoke["unique_requested"] <= CONFIG.fitness_eval_budget

# The cache key shape contains no optimiser seed; seed changes can reuse a deterministic mask score.
sample_cache_key = next(iter(smoke_cache))
assert len(sample_cache_key) == 6 and 101 not in sample_cache_key
print("Optimiser smoke tests passed.")
display(pd.DataFrame([ga_smoke, pso_smoke]).drop(columns=["mask", "features", "convergence"]))

## 13. Final inner tuning and untouched outer-test prediction

Feature selection is completed first. The chosen original-feature subset is then tuned on frozen inner folds and refitted on all outer-training data. SVM uses its decision function during selection/tuning and a sigmoid calibrator fitted only within outer-training data to create probabilities.

In [ ]:
def tune_and_evaluate(
    X_train, y_train, X_test, selected_features, inner_splits,
    classifier_name, config: Config,
) -> dict:
    """Tune inside outer training, refit and predict untouched outer test data."""
    estimator = get_base_estimator(classifier_name, config.random_state)
    pipeline = build_pipeline(classifier_name, selected_features, estimator)
    search = GridSearchCV(
        pipeline, get_param_grid(classifier_name, config.fast_mode),
        scoring="roc_auc", cv=inner_splits, refit=True,
        n_jobs=1, return_train_score=False, error_score="raise",
    )
    start = time.perf_counter()
    with capture_warnings("13", f"GridSearchCV:{classifier_name}"):
        search.fit(X_train[selected_features], y_train)
    if classifier_name == "SVM":
        calibrated = CalibratedClassifierCV(
            estimator=clone(search.best_estimator_), method="sigmoid",
            cv=inner_splits, n_jobs=1, ensemble=False,
        )
        with capture_warnings("13", "SVM sigmoid calibration"):
            calibrated.fit(X_train[selected_features], y_train)
        probabilities = calibrated.predict_proba(X_test[selected_features])[:, 1]
        decision_scores = search.best_estimator_.decision_function(X_test[selected_features])
        fitted_model = calibrated
    else:
        probabilities = search.best_estimator_.predict_proba(X_test[selected_features])[:, 1]
        decision_scores = probabilities.copy()
        fitted_model = search.best_estimator_
    return {
        "probabilities": np.asarray(probabilities),
        "decision_scores": np.asarray(decision_scores),
        "best_params": search.best_params_,
        "best_inner_auc": float(search.best_score_),
        "runtime_seconds": time.perf_counter() - start,
        "fitted_model": fitted_model,
        "calibration_method": "sigmoid" if classifier_name == "SVM" else None,
        "calibration_ensemble": False if classifier_name == "SVM" else None,
        "calibration_training_hash": index_hash(X_train.index.to_numpy()) if classifier_name == "SVM" else None,
    }


def compute_binary_metrics(y_true, probabilities, threshold: float = 0.50) -> dict:
    """Calculate discrimination, threshold and calibration metrics safely."""
    y_true = np.asarray(y_true, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    predicted = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predicted, labels=[0, 1]).ravel()
    metrics = {
        "roc_auc": roc_auc_score(y_true, probabilities) if np.unique(y_true).size == 2 else np.nan,
        "pr_auc": average_precision_score(y_true, probabilities) if np.unique(y_true).size == 2 else np.nan,
        "mcc": matthews_corrcoef(y_true, predicted),
        "sensitivity": tp / (tp + fn) if tp + fn else np.nan,
        "specificity": tn / (tn + fp) if tn + fp else np.nan,
        "f1": f1_score(y_true, predicted, zero_division=0),
        "accuracy": accuracy_score(y_true, predicted),
        "brier": brier_score_loss(y_true, probabilities),
    }
    try:
        clipped = np.clip(probabilities, 1e-6, 1 - 1e-6)
        calibration_model = LogisticRegression(C=np.inf, solver="lbfgs", max_iter=2000)
        with capture_warnings("15", "calibration intercept and slope", "analytical"):
            calibration_model.fit(logit(clipped).reshape(-1, 1), y_true)
        metrics["calibration_intercept"] = float(calibration_model.intercept_[0])
        metrics["calibration_slope"] = float(calibration_model.coef_[0, 0])
        metrics["calibration_status"] = "estimated"
    except Exception as exc:
        metrics["calibration_intercept"] = np.nan
        metrics["calibration_slope"] = np.nan
        metrics["calibration_status"] = f"not estimable: {type(exc).__name__}"
    return metrics

## 14. Main repeated nested-CV experiment

Each outer split freezes one inner split list. Within each classifier, ALL is tuned, all GA seeds are retained and the chosen GA subset is tuned, then all PSO seeds are retained and the chosen PSO subset is tuned. A complete outer-split-by-classifier unit is checkpointed with the configuration fingerprint.

In [ ]:
def save_checkpoint(unit: dict, path: Path) -> None:
    """Save a completed outer-split-by-classifier unit."""
    save_json(unit, path)


def load_checkpoint(path: Path, fingerprint: str):
    """Load only a complete compatible checkpoint."""
    if not path.exists():
        return None
    data = json.loads(path.read_text(encoding="utf-8"))
    if data.get("config_fingerprint") != fingerprint or not data.get("complete"):
        return None
    return data


prediction_rows = []
fold_metric_rows = []
optimiser_runs = []
selected_mask_rows = []
tuning_rows = []

splits_for_run = outer_splits[:1] if CONFIG.fast_mode else outer_splits
for split in splits_for_run:
    split_id = split["split_id"]
    train_idx, test_idx = split["train_idx"], split["test_idx"]
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    inner_splits = make_inner_splits(y_train, CONFIG)

    for classifier in CLASSIFIERS:
        checkpoint_path = CHECKPOINT_DIR / f"split_{split_id:02d}_{classifier}.json"
        checkpoint = load_checkpoint(checkpoint_path, CONFIG_FINGERPRINT)
        if checkpoint is not None:
            prediction_rows.extend(checkpoint["predictions"])
            fold_metric_rows.extend(checkpoint["fold_metrics"])
            optimiser_runs.extend(checkpoint["optimiser_runs"])
            selected_mask_rows.extend(checkpoint["selected_masks"])
            tuning_rows.extend(checkpoint["tuning"])
            continue

        unit_predictions, unit_metrics = [], []
        unit_runs, unit_masks, unit_tuning = [], [], []
        cache = {}
        condition_masks = {"ALL": tuple([1] * len(FEATURES))}

        ga_runs = [
            run_ga(X_train, y_train, inner_splits, classifier, split_id, seed, cache, CONFIG)
            for seed in CONFIG.optimiser_seeds
        ]
        pso_runs = [
            run_binary_pso(X_train, y_train, inner_splits, classifier, split_id, seed, cache, CONFIG)
            for seed in CONFIG.optimiser_seeds
        ]
        chosen_ga = choose_best_run(ga_runs, CONFIG.fitness_tolerance)
        chosen_pso = choose_best_run(pso_runs, CONFIG.fitness_tolerance)
        condition_masks.update({"GA": chosen_ga["mask"], "PSO": chosen_pso["mask"]})
        unit_runs.extend(ga_runs + pso_runs)

        for condition in CONDITIONS:
            mask = condition_masks[condition]
            selected = mask_to_features(mask)
            fitted = tune_and_evaluate(
                X_train, y_train, X_test, selected, inner_splits, classifier, CONFIG,
            )
            probabilities = fitted["probabilities"]
            predicted = (probabilities >= CONFIG.decision_threshold).astype(int)
            metrics = compute_binary_metrics(y_test, probabilities, CONFIG.decision_threshold)
            metric_row = {
                "outer_split": split_id, "repeat_id": split["repeat_id"],
                "fold_id": split["fold_id"], "test_hash": split["test_hash"],
                "classifier": classifier, "condition": condition,
                "feature_count": len(selected), **metrics,
            }
            unit_metrics.append(metric_row)
            unit_masks.append({
                "outer_split": split_id, "classifier": classifier, "condition": condition,
                "mask": mask, "features": selected, "feature_count": len(selected),
                "selected_by_inner_only": True,
            })
            unit_tuning.append({
                "outer_split": split_id, "classifier": classifier, "condition": condition,
                "best_inner_auc": fitted["best_inner_auc"],
                "best_params": fitted["best_params"],
                "runtime_seconds": fitted["runtime_seconds"],
                "calibration_method": fitted["calibration_method"],
                "calibration_ensemble": fitted["calibration_ensemble"],
                "calibration_training_hash": fitted["calibration_training_hash"],
            })
            for local_position, row_id in enumerate(test_idx):
                unit_predictions.append({
                    "row_id": int(row_id), "outer_split": split_id,
                    "repeat_id": split["repeat_id"], "fold_id": split["fold_id"],
                    "classifier": classifier, "condition": condition,
                    "y_true": int(y_test.iloc[local_position]),
                    "probability": float(probabilities[local_position]),
                    "predicted_class": int(predicted[local_position]),
                    "decision_score": float(fitted["decision_scores"][local_position]),
                    "mask_id": "".join(map(str, mask)),
                    "test_hash": split["test_hash"],
                })

        unit = {
            "config_fingerprint": CONFIG_FINGERPRINT, "complete": True,
            "outer_split": split_id, "classifier": classifier,
            "predictions": unit_predictions, "fold_metrics": unit_metrics,
            "optimiser_runs": unit_runs, "selected_masks": unit_masks,
            "tuning": unit_tuning,
        }
        save_checkpoint(unit, checkpoint_path)
        prediction_rows.extend(unit_predictions)
        fold_metric_rows.extend(unit_metrics)
        optimiser_runs.extend(unit_runs)
        selected_mask_rows.extend(unit_masks)
        tuning_rows.extend(unit_tuning)
        print(f"Completed split {split_id}, {classifier}")

predictions = pd.DataFrame(prediction_rows).drop_duplicates(
    ["row_id", "outer_split", "classifier", "condition"], keep="last"
)
fold_metrics = pd.DataFrame(fold_metric_rows).drop_duplicates(
    ["outer_split", "classifier", "condition"], keep="last"
)
optimiser_runs_df = pd.DataFrame(optimiser_runs).drop_duplicates(
    ["outer_split", "classifier", "selector", "seed"], keep="last"
)
selected_masks = pd.DataFrame(selected_mask_rows).drop_duplicates(
    ["outer_split", "classifier", "condition"], keep="last"
)
tuning_results = pd.DataFrame(tuning_rows).drop_duplicates(
    ["outer_split", "classifier", "condition"], keep="last"
)

predictions.to_csv(OUTPUT_DIR / "all_outer_predictions.csv", index=False)
fold_metrics.to_csv(OUTPUT_DIR / "fold_metrics.csv", index=False)
optimiser_runs_df.to_json(OUTPUT_DIR / "all_optimiser_runs.json", orient="records", indent=2)
selected_masks.to_json(OUTPUT_DIR / "selected_masks.json", orient="records", indent=2)
tuning_results.to_json(OUTPUT_DIR / "tuning_results.json", orient="records", indent=2)
pd.DataFrame(
    FITNESS_ERRORS,
    columns=["outer_split", "classifier", "mask", "error_type", "message"],
).to_csv(OUTPUT_DIR / "fitness_errors.csv", index=False)
display(fold_metrics.head(15))

## 15. Descriptive fold-level performance

Outer-fold means and standard deviations are descriptive because repeated cross-validation scores are correlated. They are not treated as independent observations, and no ordinary t-test or naïve fold-based confidence interval is used (Bengio and Grandvalet, 2004).

In [ ]:
METRIC_COLUMNS = [
    "roc_auc", "pr_auc", "mcc", "sensitivity", "specificity", "f1",
    "accuracy", "brier", "calibration_intercept", "calibration_slope",
]
fold_summary = (
    fold_metrics.groupby(["classifier", "condition"])[METRIC_COLUMNS]
    .agg(["mean", "std"]).reset_index()
)
fold_summary.columns = [
    "_".join([str(part) for part in column if part]).rstrip("_")
    if isinstance(column, tuple) else column
    for column in fold_summary.columns
]
fold_summary.to_csv(TABLE_DIR / "descriptive_outer_fold_performance.csv", index=False)
display(fold_summary)

## 16. Aggregated repeated out-of-fold predictions

For final mode, each observation has one prediction in each repeat. Probabilities are averaged by observation, classifier and condition before overall metrics, curves or confusion matrices are produced. FAST mode contains one outer fold only and is explicitly labelled incomplete.

In [ ]:
def aggregate_repeated_oof_predictions(prediction_frame: pd.DataFrame) -> pd.DataFrame:
    """Average repeated OOF probabilities to one aligned row per observation and condition."""
    grouped = prediction_frame.groupby(
        ["row_id", "classifier", "condition"], as_index=False
    ).agg(
        y_true=("y_true", "first"),
        probability=("probability", "mean"),
        prediction_count=("probability", "size"),
    )
    grouped["predicted_class"] = (grouped["probability"] >= CONFIG.decision_threshold).astype(int)
    if not CONFIG.fast_mode:
        assert grouped["prediction_count"].eq(CONFIG.outer_repeats).all()
    assert not grouped.duplicated(["row_id", "classifier", "condition"]).any()
    return grouped


aggregated_predictions = aggregate_repeated_oof_predictions(predictions)
aggregated_predictions.to_csv(OUTPUT_DIR / "aggregated_oof_predictions.csv", index=False)

aggregated_metric_rows = []
for (classifier, condition), group in aggregated_predictions.groupby(["classifier", "condition"]):
    aggregated_metric_rows.append({
        "classifier": classifier, "condition": condition,
        "n_observations": len(group),
        **compute_binary_metrics(group["y_true"], group["probability"], CONFIG.decision_threshold),
    })
aggregated_metrics = pd.DataFrame(aggregated_metric_rows)
aggregated_metrics.to_csv(TABLE_DIR / "aggregated_oof_performance.csv", index=False)
display(aggregated_metrics)

## 17. Paired, outcome-stratified bootstrap differences

For each classifier, identical within-class resampled observation indices are applied to both conditions. The metric difference is calculated inside each replicate for GA−ALL, PSO−ALL and GA−PSO. Percentile intervals quantify sampling variation conditional on the generated aggregated OOF predictions; they do not capture all model-development uncertainty.

In [ ]:
def bootstrap_metric(y_true, probabilities, metric_name: str, threshold: float):
    """Calculate one named metric from probabilities."""
    y_true = np.asarray(y_true, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    if metric_name == "roc_auc":
        return roc_auc_score(y_true, probabilities)
    if metric_name == "pr_auc":
        return average_precision_score(y_true, probabilities)
    if metric_name == "brier":
        return brier_score_loss(y_true, probabilities)
    if metric_name == "mcc":
        predicted = (probabilities >= threshold).astype(int)
        return matthews_corrcoef(y_true, predicted)
    raise ValueError(f"Unsupported bootstrap metric: {metric_name}")


def paired_stratified_bootstrap(
    aligned: pd.DataFrame, condition_a: str, condition_b: str,
    metric_name: str, replicates: int, seed: int, threshold: float,
) -> dict:
    """Bootstrap an aligned condition-A minus condition-B metric difference."""
    required = {"row_id", "y_true", condition_a, condition_b}
    if not required.issubset(aligned.columns):
        raise ValueError(f"Aligned frame lacks: {required - set(aligned.columns)}")
    y_values = aligned["y_true"].to_numpy(dtype=int)
    prob_a = aligned[condition_a].to_numpy(dtype=float)
    prob_b = aligned[condition_b].to_numpy(dtype=float)
    observed = bootstrap_metric(y_values, prob_a, metric_name, threshold) - bootstrap_metric(
        y_values, prob_b, metric_name, threshold
    )
    rng = np.random.default_rng(seed)
    class_indices = [np.flatnonzero(y_values == label) for label in [0, 1]]
    differences = []
    skipped = 0
    for _ in range(replicates):
        sampled = np.concatenate([
            rng.choice(indices, size=len(indices), replace=True) for indices in class_indices
        ])
        rng.shuffle(sampled)
        try:
            value_a = bootstrap_metric(y_values[sampled], prob_a[sampled], metric_name, threshold)
            value_b = bootstrap_metric(y_values[sampled], prob_b[sampled], metric_name, threshold)
            difference = value_a - value_b
            if np.isfinite(difference):
                differences.append(difference)
            else:
                skipped += 1
        except Exception:
            skipped += 1
    if not differences:
        lower = upper = np.nan
    else:
        lower, upper = np.percentile(differences, [2.5, 97.5])
    return {
        "condition_a": condition_a, "condition_b": condition_b,
        "comparison": f"{condition_a} - {condition_b}", "metric": metric_name,
        "observed_difference": float(observed), "ci_2_5": float(lower),
        "ci_97_5": float(upper), "valid_replicates": len(differences),
        "skipped_replicates": skipped, "bootstrap_seed": seed,
        "interval_scope": "conditional on aggregated OOF predictions",
    }


bootstrap_rows = []
comparisons_to_run = [("GA", "ALL"), ("PSO", "ALL"), ("GA", "PSO")]
bootstrap_metrics = ["roc_auc", "pr_auc", "mcc", "brier"]
for classifier in CLASSIFIERS:
    subset = aggregated_predictions.query("classifier == @classifier")
    aligned = subset.pivot(index=["row_id", "y_true"], columns="condition", values="probability").reset_index()
    if not set(CONDITIONS).issubset(aligned.columns):
        continue
    for metric_name in bootstrap_metrics:
        for condition_a, condition_b in comparisons_to_run:
            result = paired_stratified_bootstrap(
                aligned, condition_a, condition_b, metric_name,
                CONFIG.bootstrap_replicates, CONFIG.random_state, CONFIG.decision_threshold,
            )
            bootstrap_rows.append({"classifier": classifier, **result})

paired_bootstrap = pd.DataFrame(bootstrap_rows)
paired_bootstrap.to_csv(TABLE_DIR / "paired_bootstrap_differences.csv", index=False)
display(paired_bootstrap.query("metric == 'roc_auc'"))

## 18. Parsimony and stability

All retained optimiser-seed masks contribute to within-selector/classifier stability. Selection frequency and pairwise Jaccard are accompanied by the chance-corrected Nogueira estimator. Undefined cases return `NaN` with an explanatory status rather than a fabricated value.

Reference: Nogueira, Sechidis and Brown (2018), *Journal of Machine Learning Research*, 18(174), 1–54.

In [ ]:
def pairwise_jaccard(mask_matrix) -> list[float]:
    """Return pairwise Jaccard overlaps for binary feature masks."""
    matrix = np.asarray(mask_matrix, dtype=int)
    values = []
    for first, second in combinations(matrix, 2):
        union = np.logical_or(first, second).sum()
        values.append(np.logical_and(first, second).sum() / union if union else np.nan)
    return values


def nogueira_stability(mask_matrix) -> tuple[float, str]:
    """Estimate chance-corrected feature-selection stability."""
    matrix = np.asarray(mask_matrix, dtype=int)
    if matrix.ndim != 2 or matrix.shape[1] != len(FEATURES):
        return np.nan, "invalid mask matrix shape"
    if not np.isin(matrix, [0, 1]).all():
        return np.nan, "non-binary mask matrix"
    n_runs, n_features = matrix.shape
    if n_runs < 2:
        return np.nan, "fewer than two retained runs"
    selection_probability = matrix.mean(axis=0)
    unbiased_variance = n_runs / (n_runs - 1) * selection_probability * (1 - selection_probability)
    mean_selected_proportion = matrix.sum(axis=1).mean() / n_features
    denominator = mean_selected_proportion * (1 - mean_selected_proportion)
    if np.isclose(denominator, 0):
        return np.nan, "zero chance-variance denominator"
    value = 1 - unbiased_variance.mean() / denominator
    return float(value), "estimated"


# Unit checks: identical non-degenerate masks are perfectly stable; one run is undefined.
known = np.tile(np.array([[1, 0] * 6 + [1]]), (4, 1))
assert np.isclose(nogueira_stability(known)[0], 1.0)
assert np.isnan(nogueira_stability(known[:1])[0])

frequency_rows, stability_rows = [], []
for (classifier, selector), group in optimiser_runs_df.groupby(["classifier", "selector"]):
    matrix = np.vstack(group["mask"].apply(np.asarray))
    for feature, frequency in zip(FEATURES, matrix.mean(axis=0)):
        frequency_rows.append({
            "classifier": classifier, "selector": selector,
            "feature": feature, "selection_frequency": float(frequency),
        })
    jaccard = np.asarray(pairwise_jaccard(matrix), dtype=float)
    nogueira, status = nogueira_stability(matrix)
    stability_rows.append({
        "classifier": classifier, "selector": selector, "retained_runs": len(matrix),
        "jaccard_mean": float(np.nanmean(jaccard)) if jaccard.size else np.nan,
        "jaccard_median": float(np.nanmedian(jaccard)) if jaccard.size else np.nan,
        "jaccard_min": float(np.nanmin(jaccard)) if jaccard.size else np.nan,
        "jaccard_max": float(np.nanmax(jaccard)) if jaccard.size else np.nan,
        "nogueira_stability": nogueira, "nogueira_status": status,
    })

feature_frequency = pd.DataFrame(frequency_rows)
stability_summary = pd.DataFrame(stability_rows)

# Cross-classifier consensus is kept separate from within-classifier stability.
cross_classifier_consensus = (
    feature_frequency.groupby(["selector", "feature"], as_index=False)
    ["selection_frequency"].mean()
    .rename(columns={"selection_frequency": "mean_cross_classifier_frequency"})
)

overlap_rows = []
for (split_id, classifier), group in selected_masks.query(
    "condition in ['GA', 'PSO']"
).groupby(["outer_split", "classifier"]):
    masks = {row["condition"]: np.asarray(row["mask"], dtype=int) for _, row in group.iterrows()}
    if set(masks) == {"GA", "PSO"}:
        intersection = int(np.logical_and(masks["GA"], masks["PSO"]).sum())
        union = int(np.logical_or(masks["GA"], masks["PSO"]).sum())
        overlap_rows.append({
            "outer_split": split_id, "classifier": classifier,
            "intersection_features": intersection, "union_features": union,
            "ga_pso_jaccard": intersection / union if union else np.nan,
        })
ga_pso_overlap = pd.DataFrame(overlap_rows)

feature_frequency.to_csv(TABLE_DIR / "feature_selection_frequencies.csv", index=False)
stability_summary.to_csv(TABLE_DIR / "stability_jaccard_nogueira.csv", index=False)
cross_classifier_consensus.to_csv(TABLE_DIR / "cross_classifier_consensus.csv", index=False)
ga_pso_overlap.to_csv(TABLE_DIR / "ga_pso_overlap.csv", index=False)
display(stability_summary, ga_pso_overlap)

## 19. Computational fairness

The unique mask-request cap is the primary effort-control variable. Runtime is reported but depends on hardware, estimator complexity, caching and implementation. GA generations and PSO iterations are not claimed to be equivalent.

In [ ]:
efficiency_columns = [
    "outer_split", "classifier", "selector", "seed", "fitness",
    "feature_count", "runtime_seconds", "attempted_evaluations",
    "unique_requested", "cache_hits", "cache_misses", "termination",
]
efficiency_table = optimiser_runs_df[efficiency_columns].copy()
efficiency_table["maximum_unique_budget"] = CONFIG.fitness_eval_budget
efficiency_table.to_csv(TABLE_DIR / "optimiser_efficiency.csv", index=False)
display(efficiency_table)

## 20. Dissertation-ready result tables

Full-precision CSV files remain the analytical source. Display tables are rounded copies only and never overwrite source results.

In [ ]:
paired_roc = paired_bootstrap.query("metric == 'roc_auc'").copy()
parsimony = (
    selected_masks.groupby(["classifier", "condition"])["feature_count"]
    .agg(["mean", "std", "median", "min", "max"]).reset_index()
)
calibration_results = aggregated_metrics[[
    "classifier", "condition", "brier", "calibration_intercept",
    "calibration_slope", "calibration_status",
]].copy()

parsimony.to_csv(TABLE_DIR / "selected_feature_counts.csv", index=False)
calibration_results.to_csv(TABLE_DIR / "calibration_results.csv", index=False)
tuning_results.to_json(TABLE_DIR / "best_hyperparameters_by_outer_split.json", orient="records", indent=2)

display_table = aggregated_metrics.copy()
for column in METRIC_COLUMNS:
    if column in display_table:
        display_table[column] = display_table[column].round(3)
display_table.to_csv(TABLE_DIR / "display_aggregated_performance_rounded.csv", index=False)
display(display_table, paired_roc.round(3), parsimony.round(3))

## 21. Results figures

Figures are derived from saved analytical tables at full precision. Plotting is separated from model fitting, warnings are captured centrally, and Figure 08 uses a consolidated legend and constrained layout. Regenerating a figure from locked histories is a presentation operation and must never rerun optimisation.


In [ ]:
condition_palette = {"ALL": "#4C78A8", "GA": "#F58518", "PSO": "#54A24B"}

# Fold-level ROC-AUC distribution (descriptive)
plt.figure(figsize=(10, 5))
sns.boxplot(
    data=fold_metrics, x="classifier", y="roc_auc", hue="condition",
    hue_order=CONDITIONS, palette=condition_palette,
)
plt.title("Descriptive outer-fold ROC-AUC")
plt.ylabel("ROC-AUC")
with capture_warnings("21", "figure layout", "presentation"):
    plt.tight_layout()
plt.savefig(FIGURE_DIR / "02_fold_roc_auc_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

# Aggregated ROC and PR curves
fig, axes = plt.subplots(len(CLASSIFIERS), 2, figsize=(12, 4 * len(CLASSIFIERS)))
for row, classifier in enumerate(CLASSIFIERS):
    for condition in CONDITIONS:
        group = aggregated_predictions.query(
            "classifier == @classifier and condition == @condition"
        )
        if group.empty or group["y_true"].nunique() < 2:
            continue
        fpr, tpr, _ = roc_curve(group["y_true"], group["probability"])
        precision, recall, _ = precision_recall_curve(group["y_true"], group["probability"])
        auc = roc_auc_score(group["y_true"], group["probability"])
        ap = average_precision_score(group["y_true"], group["probability"])
        axes[row, 0].plot(fpr, tpr, label=f"{condition} ({auc:.3f})", color=condition_palette[condition])
        axes[row, 1].plot(recall, precision, label=f"{condition} ({ap:.3f})", color=condition_palette[condition])
    axes[row, 0].plot([0, 1], [0, 1], "--", color="grey", linewidth=1)
    axes[row, 0].set(title=f"{classifier}: ROC", xlabel="False-positive rate", ylabel="True-positive rate")
    axes[row, 1].set(title=f"{classifier}: precision-recall", xlabel="Recall", ylabel="Precision")
    axes[row, 0].legend()
    axes[row, 1].legend()
with capture_warnings("21", "figure layout", "presentation"):
    fig.tight_layout()
fig.savefig(FIGURE_DIR / "03_aggregated_roc_pr_curves.png", dpi=300, bbox_inches="tight")
plt.show()

# Calibration curves
fig, axes = plt.subplots(1, len(CLASSIFIERS), figsize=(20, 4), sharex=True, sharey=True)
for axis, classifier in zip(axes, CLASSIFIERS):
    for condition in CONDITIONS:
        group = aggregated_predictions.query(
            "classifier == @classifier and condition == @condition"
        )
        if group.empty:
            continue
        observed, predicted = calibration_curve(
            group["y_true"], group["probability"], n_bins=5, strategy="quantile",
        )
        axis.plot(predicted, observed, marker="o", label=condition, color=condition_palette[condition])
    axis.plot([0, 1], [0, 1], "--", color="grey", linewidth=1)
    axis.set(title=classifier, xlabel="Mean predicted probability")
axes[0].set_ylabel("Observed proportion")
axes[-1].legend()
fig.suptitle("Aggregated OOF calibration (secondary)", y=1.03)
with capture_warnings("21", "figure layout", "presentation"):
    fig.tight_layout()
fig.savefig(FIGURE_DIR / "04_calibration_curves.png", dpi=300, bbox_inches="tight")
plt.show()

# Paired ROC-AUC differences
forest = paired_roc.copy().sort_values(["classifier", "comparison"])
forest["label"] = forest["classifier"] + ": " + forest["comparison"]
y_positions = np.arange(len(forest))
plt.figure(figsize=(9, max(5, 0.35 * len(forest))))
plt.errorbar(
    forest["observed_difference"], y_positions,
    xerr=[forest["observed_difference"] - forest["ci_2_5"], forest["ci_97_5"] - forest["observed_difference"]],
    fmt="o", color="#4C78A8", ecolor="#9ECAE9", capsize=3,
)
plt.axvline(0, color="black", linestyle="--", linewidth=1)
plt.yticks(y_positions, forest["label"])
plt.xlabel("Paired ROC-AUC difference (95% bootstrap CI)")
plt.title("Within-classifier paired differences")
with capture_warnings("21", "figure layout", "presentation"):
    plt.tight_layout()
plt.savefig(FIGURE_DIR / "05_paired_roc_auc_forest.png", dpi=300, bbox_inches="tight")
plt.show()

# Feature selection heatmaps
for selector in SELECTORS:
    heat = feature_frequency.query("selector == @selector").pivot(
        index="classifier", columns="feature", values="selection_frequency"
    ).reindex(index=CLASSIFIERS, columns=FEATURES)
    plt.figure(figsize=(12, 4))
    sns.heatmap(heat, vmin=0, vmax=1, cmap="viridis", annot=True, fmt=".2f")
    plt.title(f"{selector} original-feature selection frequency")
    with capture_warnings("21", "figure layout", "presentation"):
        plt.tight_layout()
    plt.savefig(FIGURE_DIR / f"06_{selector.lower()}_selection_frequency.png", dpi=300, bbox_inches="tight")
    plt.show()

# Parsimony, stability and efficiency
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
sns.barplot(data=selected_masks.query("condition != 'ALL'"), x="classifier", y="feature_count", hue="condition", ax=axes[0, 0])
axes[0, 0].set_title("Selected original-feature count")
sns.barplot(data=stability_summary, x="classifier", y="nogueira_stability", hue="selector", ax=axes[0, 1])
axes[0, 1].set_title("Nogueira stability (where estimable)")
sns.barplot(data=efficiency_table, x="classifier", y="runtime_seconds", hue="selector", ax=axes[1, 0])
axes[1, 0].set_title("Optimiser runtime")
sns.barplot(data=efficiency_table, x="classifier", y="unique_requested", hue="selector", ax=axes[1, 1])
axes[1, 1].axhline(CONFIG.fitness_eval_budget, color="black", linestyle="--", linewidth=1)
axes[1, 1].set_title("Unique mask requests and common cap")
with capture_warnings("21", "figure layout", "presentation"):
    fig.tight_layout()
fig.savefig(FIGURE_DIR / "07_parsimony_stability_runtime.png", dpi=300, bbox_inches="tight")
plt.show()

# Convergence histories: line colour identifies seed; alpha separates outer splits.
seed_colours = {101: "#4C78A8", 202: "#F58518", 303: "#54A24B"}
fig, axes = plt.subplots(
    len(CLASSIFIERS), 2, figsize=(15, 17), sharex="col",
    constrained_layout=True, squeeze=False,
)
for row, classifier in enumerate(CLASSIFIERS):
    for col, selector in enumerate(SELECTORS):
        axis = axes[row, col]
        group = optimiser_runs_df.query(
            "classifier == @classifier and selector == @selector"
        ).sort_values(["seed", "outer_split"])
        for _, run in group.iterrows():
            seed = int(run["seed"])
            label = f"seed {seed}" if int(run["outer_split"]) == group["outer_split"].min() else None
            axis.plot(
                run["convergence"], color=seed_colours[seed], alpha=0.35,
                linewidth=1.1, label=label,
            )
        axis.set_title(f"{classifier} — {selector}")
        axis.set_xlabel("Generation / iteration")
        axis.set_ylabel("Best inner ROC-AUC")
        axis.grid(alpha=0.25)

handles = [plt.Line2D([0], [0], color=seed_colours[seed], label=f"seed {seed}") for seed in CONFIG.optimiser_seeds]
fig.legend(handles=handles, loc="outside upper center", ncol=len(handles), title="Optimiser seed")
fig.suptitle("Optimiser convergence across outer splits", fontsize=15)
convergence_path = FIGURE_DIR / "08_optimiser_convergence.png"
with capture_warnings("21", "Figure 08 save/layout", "presentation"):
    fig.savefig(convergence_path, dpi=300, bbox_inches="tight")
plt.show()
assert convergence_path.exists() and convergence_path.stat().st_size > 0


## 22. Interpretation-support summary

This table places performance differences, uncertainty, parsimony, stability and computation together. It does not automatically declare a winner from one metric or p-value.

In [ ]:
roc_wide = paired_roc.pivot(
    index="classifier", columns="comparison",
    values=["observed_difference", "ci_2_5", "ci_97_5"],
)
roc_wide.columns = [f"roc_auc_{stat}_{comparison.replace(' ', '_')}" for stat, comparison in roc_wide.columns]
roc_wide = roc_wide.reset_index()

selector_summary = (
    selected_masks.query("condition != 'ALL'")
    .groupby(["classifier", "condition"], as_index=False)["feature_count"].mean()
    .rename(columns={"condition": "selector", "feature_count": "mean_feature_count"})
    .merge(stability_summary, on=["classifier", "selector"], how="left")
    .merge(
        efficiency_table.groupby(["classifier", "selector"], as_index=False)
        .agg(mean_runtime_seconds=("runtime_seconds", "mean"), mean_unique_requests=("unique_requested", "mean")),
        on=["classifier", "selector"], how="left",
    )
)
selector_wide = selector_summary.pivot_table(
    index="classifier", columns="selector",
    values=[
        "mean_feature_count", "nogueira_stability", "jaccard_mean",
        "mean_runtime_seconds", "mean_unique_requests",
    ],
    aggfunc="first",
)
selector_wide.columns = [f"{measure}_{selector}" for measure, selector in selector_wide.columns]
selector_wide = selector_wide.reset_index()
interpretation_support = roc_wide.merge(selector_wide, on="classifier", how="left")
interpretation_support.to_csv(TABLE_DIR / "interpretation_support_summary.csv", index=False)
display(interpretation_support)

## 23. Automated integrity assertions and method-to-code audit

The checks below reconcile keys, counts, masks, probabilities, modes and final-run dimensions. They are recorded workflow assertions. The accompanying method-to-code table documents fitting boundaries; neither constitutes an independent forensic proof that leakage is impossible.


In [ ]:
def run_integrity_checks() -> pd.DataFrame:
    """Return recorded workflow assertions; this is not an independent code audit."""
    checks = []
    def add(name, passed, critical=True, note=""):
        checks.append({"check": name, "status": "PASS" if bool(passed) else "FAIL", "critical": critical, "note": note})

    add("Exactly five classifiers", set(predictions["classifier"]) == set(CLASSIFIERS))
    add("Exactly three conditions", set(predictions["condition"]) == set(CONDITIONS))
    expected_splits = 1 if CONFIG.fast_mode else 10
    add("Expected outer splits completed", predictions["outer_split"].nunique() == expected_splits)
    test_hash_counts = fold_metrics.groupby(["outer_split", "classifier"])["test_hash"].nunique()
    add("Matched outer-test indices", test_hash_counts.eq(1).all())
    add("No duplicate prediction keys", not predictions.duplicated(["row_id", "outer_split", "classifier", "condition"]).any())
    add("All masks have 13 original-feature bits", optimiser_runs_df["mask"].apply(len).eq(13).all())
    add("Every retained optimiser mask is non-empty", optimiser_runs_df["feature_count"].gt(0).all())
    add("GA/PSO respect common unique-request cap", optimiser_runs_df["unique_requested"].le(CONFIG.fitness_eval_budget).all(), note="The cap does not equalise attempted calls or model fits")
    add("Finite probabilities", np.isfinite(predictions["probability"]).all())
    add("Probabilities in [0, 1]", predictions["probability"].between(0, 1).all())
    add("Aggregated prediction keys unique", not aggregated_predictions.duplicated(["row_id", "classifier", "condition"]).any())
    add("Each aggregate uses two repeated predictions", aggregated_predictions["prediction_count"].eq(2).all() if "prediction_count" in aggregated_predictions else False)
    add("Paired comparison rows aligned", not paired_bootstrap.empty)
    add("Classifier-specific selector recorded", optimiser_runs_df.groupby(["classifier", "selector"]).size().gt(0).all())
    add("Selection and tuning restricted by function interfaces", True, note="Recorded design assertion using frozen inner indices; no independent code audit")
    add("SVM calibration restricted by fitting function", True, note="Recorded outer-training hash; no independent code audit")
    add("Cache context excludes optimiser seed", len(sample_cache_key) == 6 and 101 not in sample_cache_key)
    add("FAST output not labelled final", (CONFIG.fast_mode and RUN_NAME.startswith("fast_v2_")) or ((not CONFIG.fast_mode) and RUN_NAME.startswith("final_v2_")))
    add("Final execution mode", True, critical=False, note="FAST smoke test; final-only counts not applicable" if CONFIG.fast_mode else "fast_mode=false and final execution explicitly authorised")

    if not CONFIG.fast_mode:
        expected = {
            "Analytical observations": len(X) == 303,
            "Fold metric rows": len(fold_metrics) == 150,
            "Raw prediction rows": len(predictions) == 9090,
            "Aggregated prediction rows": len(aggregated_predictions) == 4545,
            "Optimiser runs": len(optimiser_runs_df) == 300,
            "Selected mask rows": len(selected_masks) == 150,
            "Tuning rows": len(tuning_results) == 150,
            "Paired bootstrap rows": len(paired_bootstrap) == 60,
            "Stability rows": len(stability_summary) == 10,
            "Thirty retained runs per stability row": stability_summary["retained_runs"].eq(30).all(),
            "All bootstrap rows have 2000 valid replicates": paired_bootstrap["valid_replicates"].eq(2000).all(),
            "No skipped bootstrap replicates": paired_bootstrap["skipped_replicates"].eq(0).all(),
        }
        for name, passed in expected.items():
            add(name, passed, critical=True)
    return pd.DataFrame(checks)


method_to_code_audit = pd.DataFrame([
    {"method": "Fold-local preprocessing", "function_or_stage": "build_pipeline/evaluate_mask/tune_and_evaluate", "fitting_data": "inner training or complete outer training", "independent_review": False},
    {"method": "Classifier-specific feature selection", "function_or_stage": "run_ga/run_binary_pso", "fitting_data": "outer training with frozen inner folds", "independent_review": False},
    {"method": "Condition-specific tuning", "function_or_stage": "GridSearchCV in tune_and_evaluate", "fitting_data": "outer training with frozen inner folds", "independent_review": False},
    {"method": "SVM sigmoid calibration", "function_or_stage": "CalibratedClassifierCV in tune_and_evaluate", "fitting_data": "outer training only", "independent_review": False},
])
method_to_code_audit.to_csv(OUTPUT_DIR / "method_to_code_audit.csv", index=False)

integrity_audit = run_integrity_checks()
integrity_audit.to_csv(OUTPUT_DIR / "integrity_audit.csv", index=False)
display(integrity_audit, method_to_code_audit)
critical_failures = integrity_audit.query("critical and status == 'FAIL'")
if not critical_failures.empty:
    raise RuntimeError("Critical integrity checks failed; final export is blocked.")


## 24. Structured warnings, canonical exports and detached manifest checksum

Warnings are saved after figures are produced. The manifest records provenance and canonical/alias status for every non-manifest artifact. A detached digest verifies the completed manifest, avoiding a self-referential checksum claim.


In [ ]:
for _, row in stability_summary.query("nogueira_status != 'estimated'").iterrows():
    CAPTURED_WARNINGS.append({
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "category": "UndefinedStatistic",
        "message": f"{row['classifier']} {row['selector']} Nogueira stability: {row['nogueira_status']}",
        "section": "18", "operation": "stability", "severity": "analytical",
        "disposition": "retained as undefined with reason",
    })
for _, row in aggregated_metrics.query("calibration_status != 'estimated'").iterrows():
    CAPTURED_WARNINGS.append({
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "category": "UndefinedStatistic",
        "message": f"{row['classifier']} {row['condition']} calibration: {row['calibration_status']}",
        "section": "17", "operation": "calibration", "severity": "analytical",
        "disposition": "retained as undefined with reason",
    })
if CONFIG.fast_mode:
    CAPTURED_WARNINGS.append({
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "category": "RunModeNotice", "message": "FAST-mode outputs are smoke-test diagnostics, not final results.",
        "section": "24", "operation": "export", "severity": "presentation", "disposition": "recorded",
    })
warning_payload = {
    "warnings": CAPTURED_WARNINGS,
    "warning_count": len(CAPTURED_WARNINGS),
    "fitness_error_count": len(FITNESS_ERRORS),
    "reconciled_after_figures": True,
}
save_json(warning_payload, OUTPUT_DIR / "warning_error_log.json")

manifest_path = OUTPUT_DIR / "run_manifest.json"
detached_path = OUTPUT_DIR / "run_manifest.sha256"
for stale in [manifest_path, detached_path]:
    if stale.exists():
        stale.unlink()

manifest_files = [artifact_metadata(path) for path in sorted(OUTPUT_DIR.rglob("*")) if path.is_file()]
manifest = {
    "run_name": RUN_NAME,
    "mode": "FAST smoke test" if CONFIG.fast_mode else "FINAL V2",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "configuration_fingerprint": CONFIG_FINGERPRINT,
    "self_checksum_policy": "The manifest cannot contain its own stable checksum; use run_manifest.sha256.",
    "files": manifest_files,
}
save_json(manifest, manifest_path)
detached_digest = sha256_file(manifest_path)
detached_path.write_text(f"{detached_digest}  {manifest_path.name}\n", encoding="utf-8")
assert sha256_file(manifest_path) == detached_path.read_text(encoding="utf-8").split()[0]
for item in manifest_files:
    path = OUTPUT_DIR / item["relative_path"]
    assert path.stat().st_size == item["size_bytes"] and sha256_file(path) == item["sha256"]

print("Completed 25 analytical sections.")
print(f"Run mode: {manifest['mode']}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")
print(f"Captured warnings: {len(CAPTURED_WARNINGS)}; fitness errors: {len(FITNESS_ERRORS)}")
print("Integrity rows are workflow assertions, not an independent audit.")
print("Generated canonical/alias artifacts:")
for item in manifest["files"]:
    suffix = f" -> alias of {item['alias_of']}" if item.get("alias_of") else ""
    print(" -", item["relative_path"], suffix)


## Methodological references

- Bengio, Y. and Grandvalet, Y. (2004) ‘No unbiased estimator of the variance of k-fold cross-validation’, *Journal of Machine Learning Research*, 5, 1089–1105. https://www.jmlr.org/papers/v5/grandvalet04a.html
- Cawley, G.C. and Talbot, N.L.C. (2010) ‘On over-fitting in model selection and subsequent selection bias in performance evaluation’, *Journal of Machine Learning Research*, 11, 2079–2107. https://www.jmlr.org/papers/v11/cawley10a.html
- Chen, T. and Guestrin, C. (2016) ‘XGBoost: A scalable tree boosting system’, *Proceedings of KDD 2016*, 785–794. https://doi.org/10.1145/2939672.2939785
- Collins, G.S. et al. (2024) ‘TRIPOD+AI statement’, *BMJ*, 385, e078378. https://doi.org/10.1136/bmj-2023-078378
- Kohavi, R. and John, G.H. (1997) ‘Wrappers for feature subset selection’, *Artificial Intelligence*, 97(1–2), 273–324. https://doi.org/10.1016/S0004-3702(97)00043-X
- Moons, K.G.M. et al. (2025) ‘PROBAST+AI’, *BMJ*, 388, e082505. https://doi.org/10.1136/bmj-2024-082505
- Nogueira, S., Sechidis, K. and Brown, G. (2018) ‘On the stability of feature selection algorithms’, *Journal of Machine Learning Research*, 18(174), 1–54. https://www.jmlr.org/papers/v18/17-514.html
- Steyerberg, E.W. et al. (2010) ‘Assessing the performance of prediction models’, *Epidemiology*, 21(1), 128–138. https://doi.org/10.1097/EDE.0b013e3181c30fb2
- UCI Machine Learning Repository (1988) *Heart Disease*. https://doi.org/10.24432/C52P4X
- Van Calster, B. et al. (2019) ‘Calibration: the Achilles heel of predictive analytics’, *BMC Medicine*, 17, 230. https://doi.org/10.1186/s12916-019-1466-7
- Varma, S. and Simon, R. (2006) ‘Bias in error estimation when using cross-validation for model selection’, *BMC Bioinformatics*, 7, 91. https://doi.org/10.1186/1471-2105-7-91